# ML Training Notebook
Train models, evaluate performance, track experiments:

1. DummyClassifier
2. RandomForestClassifier
3. LogisticRegression
4. MLflow tracking
5. Learning curves

In [1]:
from pathlib import Path
import os

import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


from sklearn.model_selection import learning_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

import warnings
import time
from google.cloud import storage

warnings.filterwarnings('ignore')

## Model Hyperparameter Configuration

In [2]:
# Define hyperparameters for SVM and Deep Neural Network
SVM_RANDOM_STATE = 42
SVM_MAX_ITER = 1000

# PyTorch Neural Network Hyperparameters
NN_INPUT_SIZE = 8  # Number of preprocessed features (Pclass, Age, SibSp, Parch, Fare, Sex_male, Embarked_Q, Embarked_S)
NN_HIDDEN_SIZE = 1000
NN_EPOCHS = 100
NN_BATCH_SIZE = 64
NN_LEARNING_RATE = 0.001
NN_RANDOM_STATE = 42

# Device selection
DEVICE = 'cpu' #normally it would be torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


Using device: cpu


In [3]:
# Set random seeds for reproducibility
torch.manual_seed(NN_RANDOM_STATE)
np.random.seed(NN_RANDOM_STATE)

## Load Engineered Datasets

In [13]:
# Data paths - Local
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent

print("Project Root:", PROJECT_ROOT)

df = pd.read_parquet(
    PROJECT_ROOT /
    "data" /
    "features" /
    "air_quality_features.parquet"
)

print(Path.cwd())
print(df.shape)
df.head()
df.info()

Project Root: /home/chloekc/datahandling/healthyair
/home/chloekc/datahandling/healthyair/notebooks
(9353, 61)
<class 'pandas.DataFrame'>
RangeIndex: 9353 entries, 0 to 9352
Data columns (total 61 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Date                          9353 non-null   str           
 1   Time                          9353 non-null   str           
 2   CO(GT)                        9353 non-null   float64       
 3   PT08.S1(CO)                   9353 non-null   float64       
 4   NMHC(GT)                      9353 non-null   float64       
 5   C6H6(GT)                      9353 non-null   float64       
 6   PT08.S2(NMHC)                 9353 non-null   float64       
 7   NOx(GT)                       9353 non-null   float64       
 8   PT08.S3(NOx)                  9353 non-null   float64       
 9   NO2(GT)                       9353 non-null   float64       
 

In [14]:
train_df = pd.read_parquet(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "train.parquet"
)

eval_df = pd.read_parquet(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "evaluation.parquet"
)

test_df = pd.read_parquet(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "test.parquet"
)

print(train_df.shape)
print(eval_df.shape)
print(test_df.shape)
train_df.info()

(5398, 20)
(1156, 20)
(1158, 20)
<class 'pandas.DataFrame'>
RangeIndex: 5398 entries, 0 to 5397
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              5398 non-null   str           
 1   Time              5398 non-null   str           
 2   CO(GT)            5398 non-null   float64       
 3   PT08.S1(CO)       5398 non-null   float64       
 4   NMHC(GT)          5398 non-null   float64       
 5   C6H6(GT)          5398 non-null   float64       
 6   PT08.S2(NMHC)     5398 non-null   float64       
 7   NOx(GT)           5398 non-null   float64       
 8   PT08.S3(NOx)      5398 non-null   float64       
 9   NO2(GT)           5134 non-null   float64       
 10  PT08.S4(NO2)      5398 non-null   float64       
 11  PT08.S5(O3)       5398 non-null   float64       
 12  T                 5398 non-null   float64       
 13  RH                5398 non-null   float64       
 14  AH

In [16]:
# Google Cloud Storage Configuration
# ---------------------------------------------------------------------
PROJECT_ID = "healthy-air-analysis"
BUCKET_NAME = "uci_air_quality_20119102"
LOCATION = "EU"

print(f"Project: {PROJECT_ID}")
print(f"Bucket: gs://{BUCKET_NAME}")

# Load the dataset splits from GCS processed/v1 folder
train_url = f"gs://{BUCKET_NAME}/processed/v1/train.parquet"
eval_url = f"gs://{BUCKET_NAME}/processed/v1/evaluation.parquet"
test_url = f"gs://{BUCKET_NAME}/processed/v1/test.parquet"

# Reading the files using pandas
df_train = pd.read_parquet(train_url)
df_eval = pd.read_parquet(eval_url)
df_test = pd.read_parquet(test_url)

print(f"Loaded train shape: {df_train.shape}, dev shape: {df_eval.shape}, test shape: {df_test.shape}")
df_train.info()

Project: healthy-air-analysis
Bucket: gs://uci_air_quality_20119102
Loaded train shape: (5398, 20), dev shape: (1156, 20), test shape: (1158, 20)
<class 'pandas.DataFrame'>
RangeIndex: 5398 entries, 0 to 5397
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              5398 non-null   str           
 1   Time              5398 non-null   str           
 2   CO(GT)            5398 non-null   float64       
 3   PT08.S1(CO)       5398 non-null   float64       
 4   NMHC(GT)          5398 non-null   float64       
 5   C6H6(GT)          5398 non-null   float64       
 6   PT08.S2(NMHC)     5398 non-null   float64       
 7   NOx(GT)           5398 non-null   float64       
 8   PT08.S3(NOx)      5398 non-null   float64       
 9   NO2(GT)           5134 non-null   float64       
 10  PT08.S4(NO2)      5398 non-null   float64       
 11  PT08.S5(O3)       5398 non-null   float64       
 1

## Prepare Features and Target

In [ ]:
# Impute missing values with median
numeric_cols = df_clean.select_dtypes(
    include="number"
).columns

for col in numeric_cols:

    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

In [ ]:
# Preparing features
features = ['CO(GT)','PT08.S1(CO)','NMHC(GT)','C6H6(GT)','PT08.S2(NMHC)','NOx(GT)','PT08.S3(NOx)','NO2(GT)','PT08.S4(NO2)','PT08.S5(O3)','T','RH','AH']

# Helper function to process each dataset split consistently
def preprocess_split(df_split):
    df_processed = df_split[features].copy()

    # Fill missing values using training set statistics or defaults
    df_processed['Age'].fillna(df_train['Age'].median(), inplace=True)
    df_processed['Fare'].fillna(df_train['Fare'].median(), inplace=True)
    df_processed['Embarked'].fillna(df_train['Embarked'].mode()[0], inplace=True)

    # Convert categorical features to numeric
    df_processed = pd.get_dummies(df_processed, columns=['Sex', 'Embarked'], drop_first=True)

    # Ensure both splits have the exact same columns after dummy encoding
    expected_columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']
    for col in expected_columns:
        if col not in df_processed.columns:
            df_processed[col] = 0
    return df_processed[expected_columns]

# Prepare features for both Train and Dev splits
X_train = preprocess_split(df_train)
y_train = df_train['Survived']

X_test = preprocess_split(df_dev)
y_test = df_dev['Survived']

print(f"Final shapes - X_train: {X_train.shape}, X_test (dev): {X_test.shape}")


In [ ]:
#Feature Normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Define PyTorch MLP Model Architecture
class TitanicMLP(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(TitanicMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


In [ ]:
features = exclude_cols = [
    "Datetime",
    "Target_Next_Hour",
    "Date",
    "Time"
]

#train_df = create_features(train_df)

#eval_df = create_features(eval_df)

#test_df = create_features(test_df)

In [ ]:
# Candidate predictors: current measurements only
predictors = measurement_cols.copy()

# Prepare modelling dataset
model_df = forecast_df[
    ["Datetime"] + predictors + ["Target_Next_Hour"]
].dropna(subset=["Target_Next_Hour"])

# Chronological 80/20 train-test split
split_index = int(len(model_df) * 0.8)

train = model_df.iloc[:split_index]
test = model_df.iloc[split_index:]

X_train = train[predictors]
X_test = test[predictors]

y_train = train["Target_Next_Hour"]
y_test = test["Target_Next_Hour"]

# Remove predictors completely missing in the training data
usable_predictors = X_train.columns[
    X_train.notna().any()
].tolist()

X_train = X_train[usable_predictors]
X_test = X_test[usable_predictors]

# Baseline: always predict the most frequent training category
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

# Random Forest pipeline
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_predictions = rf_pipeline.predict(X_test)

# Compare model performance
results = pd.DataFrame({
    "model": ["Baseline", "Random Forest"],
    "accuracy": [
        baseline.score(X_test, y_test),
        rf_pipeline.score(X_test, y_test)
    ],
    "balanced_accuracy": [
        balanced_accuracy_score(y_test, baseline_predictions),
        balanced_accuracy_score(y_test, rf_predictions)
    ],
    "macro_f1": [
        f1_score(y_test, baseline_predictions, average="macro"),
        f1_score(y_test, rf_predictions, average="macro")
    ]
})

display(results.round(3))

# Detailed Random Forest results
print("Random Forest classification report:")
print(classification_report(
    y_test,
    rf_predictions,
    zero_division=0
))